# Electricity Theft Detection

**Track:** Energy Systems — Predictive Maintenance / Anomaly Detection
**Advanced Topics:** XAI (SHAP) + Adversarial Robustness (FGSM)
**Dataset:** UCI Electricity Load Diagrams + Synthetic Theft Labels

In [ ]:
import subprocess
subprocess.check_call(['pip', 'install', 'shap'])
print('Dependencies installed')
# NOTE: This cell requires a Colab runtime restart after execution before proceeding to next cells.

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import shap
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support, classification_report, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print(f'PyTorch: {torch.__version__}')
print(f'SHAP: {shap.__version__}')

In [ ]:
# Install kagglehub for reliable dataset access
import subprocess
subprocess.check_call(['pip', 'install', '-q', 'kagglehub'])
print('kagglehub installed')

# Download Electricity Load Diagrams dataset via kagglehub
import kagglehub
path = kagglehub.dataset_download("eduardojst10/electricityloaddiagrams20112014")

# Read only the first meter (Meter_001) to keep Colab memory manageable
import pandas as pd
meter_path = f"{path}/Electricity/Meter_001.csv"
df = pd.read_csv(meter_path)
print(f'Shape: {df.shape}')
print(df.head())
print(df.info())

In [ ]:
# Plot load data for the first meter
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
for i in range(min(4, len(df))):
    # Show a subset of data for each subplot (different time ranges)
    start = i * len(df) // 4
    end = start + len(df) // 8
    axes[i].plot(range(start, end), df.iloc[start:end].values, linewidth=0.5)
    axes[i].set_title(f'Portion {i+1} of Meter_001 Load Diagram')
    axes[i].set_ylabel('Load (kWh)')
plt.tight_layout()
plt.savefig('baseline_load_patterns.png', dpi=150, bbox_inches='tight')
plt.show()

# Basic statistics
print(df.describe())

# Check for missing values
missing = df.isnull().sum()
print(f'Missing values per column:\n{missing[missing > 0]}')